[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/1-using-tool.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 1. Tool calling

One tool, two model calls. The model asks for `get_store_fact` and passes a topic, not a filename. This notebook sends the returns section back. The second call writes the answer from that text.

The person asking is a customer. The question is whether an opened jar can come back.

You need `OPENAI_API_KEY` in the repository-root `.env`, or in a Colab secret of the same name.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## The tool

`get_store_fact` lives in `tutorial.common.facts`. Importing it registers the tool. The model names a topic. Python maps that topic to a file and a heading. The path jail stays inside the lookup. It is not a Chat Completions tool.

In [ ]:
import tutorial.common.facts  # registers get_store_fact

from tutorial.common.harness import check, system_text
from tutorial.common.messages import assistant_message
from tutorial.common.tools import TOOLS, call_tool
from tutorial.runtime import chat

print("tools:", [tool["function"]["name"] for tool in TOOLS])

## 1. Ask

Send the customer's question and the tool list. This first call should request `get_store_fact`. It does not open the file.

In [ ]:
question = "I opened a jar of fig jam. Can I return it?"
messages = [
    {"role": "system", "content": system_text()},
    {"role": "user", "content": question},
]
print("QUESTION:", question)
reply = chat(messages, TOOLS)
print("tool_calls:", [call["name"] for call in reply["tool_calls"]])

## 2. Run the tool

The model returns a tool name as a string. `call_tool` looks that name up and runs the function with `call["arguments"]`.

In [ ]:
result = ""
if not reply["tool_calls"]:
    print("The model answered without a tool. That paragraph did not open the policy.")
else:
    call = reply["tool_calls"][0]
    messages.append(assistant_message(reply))
    result = call_tool(call)
    print(call["name"], call["arguments"])
    print(result)
    messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    topic = str(call["arguments"].get("topic", ""))
    check(call["name"] == "get_store_fact", "the model requested get_store_fact")
    check(topic == "returns", "the topic is returns")
    check("/" not in topic and not topic.endswith(".md"), "the model passed a topic, not a filename")
    check("final sale" in result.lower(), "the tool returned the opened-jar rule")

## 3. Answer

Call the model again. The reply should say an opened jar is final sale because the tool result said so.

In [ ]:
if not result:
    print("Skipped the second call because there was no tool result.")
else:
    reply = chat(messages, TOOLS)
    print("ANSWER:", reply["content"])
    check("final sale" in (reply["content"] or "").lower(), "the answer uses the returns section")